# GEARS - Notebook 3 : Scenarios moyen et long terme (GMM & VAE)

Ce notebook illustre la chaine complete de prevision et de simulation de la demande de recharge VE sur des horizons etendus, depuis les previsions departementales a 1-3 ans jusqu'aux trajectoires nationales a 5-20 ans, avec reconstruction des profils de charge horaires et optimisation de smart charging - pour le GMM **et** le VAE.

**Section A - Moyen terme (1-3 ans)**
Prevision d'energie journaliere par departement avec `DepartmentForecaster` (SARIMA) et fan charts.

**Section B - Long terme (5-20 ans)**
Trois scenarios d'adoption VE (conservateur / central / ambitieux) traduits en trajectoires d'energie journaliere avec bruit Monte-Carlo, construits directement a partir des fonctions `gears.simulation.medium_term` (et non plus de formules dupliquees a la main dans ce notebook).

**Section C - Profils de charge et smart charging (GMM + VAE)**
Reconstruction du profil horaire annuel depuis le GMM **et** le VAE (`registry.load("french_vae_sample")`) avec `OutputAggregator`, puis comparaison plug-and-charge vs V1G avec `SmartChargingOptimizer` pour les deux modeles.

> **Notes d'execution :** le fichier reel `data/sample_df.pkl` couvre ~9,5 ans (2016-2026) et 101
> departements (~2,7M sessions) - bien plus que ce dont ce notebook a besoin. Pour rester dans un budget
> de quelques minutes : (a) `FOCUS_DEPTS` est reduit a 3 departements et la fenetre d'entrainement SARIMA
> a une fenetre recente (`RECENT_MONTHS`) ; un seul `.fit()` est reutilise pour la fenetre de test ET la
> vue prospective (au lieu d'un second `.fit()` complet sur tout l'historique) ; (b) la Section C -
> normalement **>30 minutes** a cause de la reconstruction smart charging sur les 8008 contextes du
> bundle `"french"` complet - est reconstruite sur un sous-ensemble restreint de contextes (2 types de
> lieu x les departements focus), en gardant la reconstruction "plug" complete (rapide, ~20s) sur le
> bundle entier pour rester representative.
> La cle registry reelle du VAE est `"french_vae_sample"` (et non `"french_vae"`).
>
> **Session 6 :** cette version corrige deux bugs traces numeriquement (voir REFACTOR_STATE.md /
> AUDIT.md §e) qui produisaient un "plafond" artificiel dans les scenarios long terme -- un bruit de
> prevision (Section A) artificiellement pince (`DepartmentForecaster` utilisait 5% de l'ecart-type
> historique au lieu de l'ecart-type complet), et un clip d'affichage a 10x l'ancre (Section B) qui
> aplatissait visuellement les scenarios central et ambitieux bien avant l'horizon 2040. La Section B
> a egalement ete reconstruite pour appeler reellement les fonctions de profil de croissance du package
> (`linear_growth_profile`, `s_curve_growth_profile`, `bass_diffusion_profile`), corrigees pour demarrer
> exactement au niveau observe actuel (t=0) et pour que leur saturation s'etale sur tout l'horizon
> demande plutot que sur un nombre d'annees fixe. Le nombre de modeles de croissance exposes est reduit
> de 5 a 3 (voir REFACTOR_STATE.md pour la justification de chaque suppression).

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os; os.makedirs("outputs", exist_ok=True)
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import gears
from gears import NativeGMMRegistry, load_sessions
from gears import CHARGER_PRESETS, OutputAggregator, LOCATION_POWER_PRESETS, SmartChargingOptimizer
from gears.data.insee import DepartmentForecaster, aggregate_by_department, build_panel
from gears.models.gmm import EVSessionGMM
from gears.simulation.medium_term import (
    linear_growth_profile,
    s_curve_growth_profile,
    bass_diffusion_profile,
)
from gears.plotting import plot_mt_fan_charts, plot_lt_trajectories

In [ ]:
# === CONFIGURATION ===

# Donnees
DATA_PATH     = "../data/sample_df.pkl"
FOCUS_DEPTS   = ["92", "69", "78"]   # reduit de 6 a 3 departements (budget d'execution)
RECENT_MONTHS = 14                  # fenetre d'historique pour le SARIMA departemental

# Section A - Moyen terme
N_SCENARIOS      = 15    # reduit de 50 (budget d'execution)
TEST_DAYS        = 45    # reduit de 90 (fenetre de test / hold-out)
FORECAST_DAYS    = 60    # jours de prevision AU-DELA de la fenetre de test (reduit de 180)
COMBINED_HORIZON = TEST_DAYS + FORECAST_DAYS   # un seul .fit() + un seul .predict() couvrent les deux

# Section B - Long terme
N_LONG_SCENARIOS = 10    # trajectoires LT (le builder impose un minimum de 30)
LONG_YEARS       = 15    # horizon 2025 -> 2040
NATIONAL_MONTHS  = 13    # fenetre (tous departements, mois calendaires complets) pour le baseline national

# Section C - Profils de charge et smart charging
N_DAYS_MC_PROFILES = 8            # jours Monte-Carlo (reduit de 20) pour le profil "plug" (bundle complet)
N_DAYS_MC_SMART    = 5             # jours Monte-Carlo pour la reconstruction smart charging (sous-ensemble)
PROFILE_YEAR       = 2025          # annee de reference pour la reconstruction
PROFILE_PRESET     = "french_2024" # cle dans LOCATION_POWER_PRESETS
SMART_LOC_TYPES    = {"work", "home"}   # sous-ensemble de lieux pour Section C smart charging
SMART_DEPTS        = set(FOCUS_DEPTS)   # sous-ensemble de departements pour Section C smart charging

# Constantes parc VE francais (Avere-France, T4 2024)
TOTAL_FLEET          = 38_500_000   # parc total VP
EV_SHARE_NOW         = 0.031        # taux d'electrification actuel (~3,1 %)
EV_VEHICLES_NOW      = int(TOTAL_FLEET * EV_SHARE_NOW)   # ~= 1,19 M VE
SESSIONS_PER_EV_DAY  = 0.22         # ~= 80 sessions/VE/an (donnees IRVE)

print(f"Parc total  : {TOTAL_FLEET:,} vehicules")
print(f"Part VE     : {EV_SHARE_NOW:.1%}  ({EV_VEHICLES_NOW:,} VE)")
print(f"Sessions/j  : {EV_SHARE_NOW * TOTAL_FLEET * SESSIONS_PER_EV_DAY:,.0f}")

---
# Part A — Medium Term (1–3 years)

Département-level daily energy demand forecasting with `DepartmentForecaster` (SARIMA per département).

In [ ]:
# Le fichier reel couvre 101 departements sur ~9,5 ans (~2,7M sessions, ~1,3 Go une fois
# valide). On filtre sur FOCUS_DEPTS *avant* la validation pour limiter memoire et temps de
# calcul, puis on restreint a une fenetre recente (RECENT_MONTHS) : les fits SARIMA sur de tres
# longues series (des annees) sont a la fois lents et gourmands en memoire pour un gain de
# fidelite minime - le comportement de charge recent est aussi plus representatif.
_raw = pd.read_pickle(DATA_PATH)
_dept_col = "insee_code_departement" if "insee_code_departement" in _raw.columns else "department"
_raw = _raw[_raw[_dept_col].astype(str).str.strip().isin(FOCUS_DEPTS)].copy()
df = load_sessions(_raw, verbose=True)
del _raw
gc.collect()

_cutoff = df["arrival_time"].max() - pd.DateOffset(months=RECENT_MONTHS)
df = df[df["arrival_time"] >= _cutoff].copy()

print(f"\nDate range (fenetre recente): {df['arrival_time'].min().date()} -> {df['arrival_time'].max().date()}")
print(f"Departments: {df['department'].nunique()}  ({sorted(df['department'].unique())})")

In [ ]:
# Construit le panel DatetimeIndex x colonnes-departement, puis un apercu mensuel.
panel = build_panel(df, freq="D", metric="energy_kwh")
panel.index = pd.to_datetime(panel.index)   # garantit un DatetimeIndex propre
print(f"Panel shape : {panel.shape[0]} jours x {panel.shape[1]} departements")
print(f"Date range  : {panel.index.min().date()} -> {panel.index.max().date()}")
print(f"Depts focus disponibles : {[d for d in FOCUS_DEPTS if d in panel.columns]}")

agg_monthly = aggregate_by_department(df, freq="ME", metric="energy_kwh")
agg_monthly["date"] = pd.to_datetime(agg_monthly["date"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, dept in zip(np.atleast_1d(axes).flat, FOCUS_DEPTS):
    sub = agg_monthly[agg_monthly["department"] == dept].sort_values("date") \
        if "department" in agg_monthly.columns else None
    if sub is None or sub.empty:
        ax.set_visible(False); continue
    ax.plot(sub["date"].values, sub["energy_kwh"].values, color="#2E86AB", linewidth=2)
    ax.set_title(f"Departement {dept} - energie mensuelle (kWh)", fontsize=10)
    ax.set_ylabel("kWh / mois"); ax.set_xlabel("")
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.tick_params(axis="x", rotation=30)

fig.suptitle("Energie de recharge mensuelle par departement (historique recent)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.savefig("outputs/03_monthly_energy_by_dept.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Train / test split, puis fit du DepartmentForecaster (SARIMA par departement).
split_date  = panel.index.max() - pd.Timedelta(days=TEST_DAYS)
panel_train = panel[panel.index <= split_date]
panel_test  = panel[panel.index >  split_date]

train_df = df[df["arrival_time"] <= split_date].copy()
test_df  = df[df["arrival_time"] >  split_date].copy()

print(f"Train: {panel_train.index.min().date()} -> {panel_train.index.max().date()}  ({len(panel_train)} days)")
print(f"Test : {panel_test.index.min().date()} -> {panel_test.index.max().date()}  ({len(panel_test)} days)")

fc = DepartmentForecaster(
    seasonal_period=7, max_p=2, max_q=2, max_P=1, max_Q=1,
    min_obs=60, use_log=True,
)
fc.fit(train_df, departments=FOCUS_DEPTS, verbose=True)

In [ ]:
# Un seul .predict() sur COMBINED_HORIZON couvre a la fois la fenetre de test (evaluee contre
# les vraies valeurs) et la vue prospective au-dela (reutilisee telle quelle plus bas, pas de
# second .fit() complet sur tout l'historique).
full_fc = fc.predict(
    horizon=COMBINED_HORIZON,
    departments=FOCUS_DEPTS,
    n_scenarios=N_SCENARIOS,
    start_date=panel_test.index.min(),
    seed=0,
)
full_fc["date"] = pd.to_datetime(full_fc["date"])

test_end   = panel_test.index.min() + pd.Timedelta(days=TEST_DAYS)
test_fc    = full_fc[full_fc["date"] < test_end].copy()
forward_fc = full_fc   # vue prospective : reutilise le meme fit + le meme predict que ci-dessus

rows = []
for dept in FOCUS_DEPTS:
    if dept not in panel_test.columns: continue
    actual = panel_test[dept].fillna(0)
    pred_dept = test_fc[test_fc["department"] == dept]
    if pred_dept.empty: continue
    pred_median = (
        pred_dept.groupby("date")["energy_kwh_forecast"]
        .median()
        .reindex(actual.index, fill_value=0)
    )
    mae  = float(np.abs(pred_median - actual).mean())
    rmse = float(np.sqrt(((pred_median - actual)**2).mean()))
    nz   = actual > 0
    mape = float(np.abs((pred_median[nz] - actual[nz]) / actual[nz]).mean()) * 100
    rows.append({"Département": dept, "MAE (kWh)": round(mae, 0),
                 "RMSE (kWh)": round(rmse, 0), "MAPE (%)": round(mape, 1)})

metrics_df = pd.DataFrame(rows)
print("-- Metriques d'evaluation moyen terme --------------------------------")
print(metrics_df.to_string(index=False))
print(f"\nVue prospective : {COMBINED_HORIZON} jours au total pour {len(FOCUS_DEPTS)} departements "
      f"(dont les {TEST_DAYS} premiers jours sont evalues ci-dessus contre la verite terrain)")

In [ ]:
# Session 6 : DepartmentForecaster._forecast_dept utilise desormais l'ecart-type complet
# (pas 5%) comme echelle de bruit -- voir gears/data/insee.py et REFACTOR_STATE.md. La bande
# de confiance ci-dessous devrait etre nettement visible, pas une ligne quasi plate.
fig_mt = plot_mt_fan_charts(
    panel=panel_train,
    forecast_df=test_fc,
    departments=FOCUS_DEPTS,
    hist_tail_days=30,
    metrics_df=metrics_df,
    n_cols=3,
    figsize=(16, 8),
    title=f"Prevision energie moyen terme -- fenetre test ({TEST_DAYS} jours)",
    savepath="outputs/03_medium_fan_charts.png",
)

# Superposition des valeurs reelles sur les fan charts de la periode de test
axes_flat = np.array(fig_mt.axes).flatten()
for ax, dept in zip(axes_flat, FOCUS_DEPTS):
    if dept not in panel_test.columns:
        continue
    actual = panel_test[dept].fillna(0)
    ax.plot(
        actual.index, actual.values,
        color="#222222", linewidth=1.2, linestyle="--", zorder=5,
        label="Valeur reelle (test)",
    )
    ax.legend(fontsize=7, loc="lower right")

fig_mt.savefig("outputs/03_medium_fan_charts.png", dpi=150, bbox_inches="tight")
plt.show()

# Largeur mesuree de la bande 80% (utile pour confirmer le fix du bruit ci-dessus, pas
# seulement l'affirmer) :
pivot_check = test_fc.pivot_table(index="date", columns=["department", "scenario"],
                                   values="energy_kwh_forecast")
med_check = pivot_check.median(axis=1)
width_check = pivot_check.quantile(0.9, axis=1) - pivot_check.quantile(0.1, axis=1)
rel_width_check = float((width_check / med_check.clip(lower=1e-6)).mean())
print(f"Largeur moyenne de la bande 80% (mesuree) : {rel_width_check:.1%} du median "
      f"(avant Session 6, ~2% attendu -- voir AUDIT.md Mecanisme 1)")

---
# Part B — Long Term (5–20 years)

### EV market adoption curves → energy implications

**Approach**
1. Define three EV fleet adoption scenarios, built from the (Session 6-fixed)
   `linear_growth_profile` / `s_curve_growth_profile` / `bass_diffusion_profile`
   functions in `gears.simulation.medium_term` — each anchored exactly at
   today's observed fleet share at t=0, not zero.
2. Plot adoption curves and their derivative (annual new EVs added).
3. Scale the current national baseline energy by each scenario's adoption
   multiplier, with Monte-Carlo noise calibrated on real historical
   volatility, to get daily-energy trajectories.
4. Analyse grid capacity implications.

**Sources:** Avere-France, SNEF, Observatoire du Véhicule d'Entreprise (2024),
EU Green Deal 2035 target, French PNEC 2030 objectives.

In [ ]:
# Session 6: ces trois scenarios sont desormais construits en appelant reellement les
# fonctions de gears.simulation.medium_term (au lieu de dupliquer leurs formules a la main).
# Chaque fonction est appelee avec base_sessions_per_day=1.0 : sa sortie EST directement la
# trajectoire "multiple du taux actuel", puisque les trois profils garantissent maintenant
# croissance(t=0) == 1.0 exactement (corrige en Session 6 -- voir REFACTOR_STATE.md).
# Seule adoption_mul.values est utilisee (pas adoption_mul.index) : la calibration temporelle
# reelle (sim_start_date) n'est connue qu'une fois panel_nat charge plus bas.
START_YEAR = 2025
END_YEAR   = START_YEAR + LONG_YEARS

conservative_mul = linear_growth_profile(
    1.0, LONG_YEARS, annual_growth_rate=(0.20 / EV_SHARE_NOW - 1) / LONG_YEARS,
)
central_mul = s_curve_growth_profile(
    1.0, LONG_YEARS, saturation_factor=0.40 / EV_SHARE_NOW,
)
ambitious_mul = bass_diffusion_profile(
    1.0, LONG_YEARS, market_potential_factor=0.65 / EV_SHARE_NOW, p=0.03, q=0.38,
)

scenarios = {
    "Conservative (linear)": ("A", conservative_mul, "#9B5DE5", "-"),
    "Central (s-curve)":     ("B", central_mul,      "#2E86AB", "--"),
    "Ambitious (bass)":      ("C", ambitious_mul,    "#E84855", "-."),
}

print(f"Scenarios defined -- starting from {EV_SHARE_NOW:.1%} EV share ({EV_VEHICLES_NOW:,} vehicles)")
print(f"{'Year':>6}  {'Conservative':>14}  {'Central':>12}  {'Ambitious':>12}")
for yr_offset in [0, 2, 5, 8, 12, 15]:
    idx = min(int(yr_offset * 365.25), len(conservative_mul) - 1)
    row = [f"{START_YEAR + yr_offset:>6}"]
    for name, (_, mul, _, _) in scenarios.items():
        share = EV_SHARE_NOW * mul.iloc[idx]
        row.append(f"{share:.1%}".rjust(14 if "Conservative" in name else 12))
    print("  ".join(row))

print(f"\nGrowth factor {START_YEAR}->{END_YEAR} (share / energy-proportional; measured, not assumed):")
for name, (lbl, mul, _, _) in scenarios.items():
    factor = float(mul.iloc[-1] / mul.iloc[0])
    print(f"  {name:24s}: {factor:5.1f}x")

In [ ]:
years_numeric = START_YEAR + np.arange(len(conservative_mul)) / 365.25

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for name, (lbl, mul, color, ls) in scenarios.items():
    ax.plot(years_numeric, EV_SHARE_NOW * mul.values * 100, color=color, linestyle=ls,
            linewidth=2.5, label=name)
ax.scatter([START_YEAR], [EV_SHARE_NOW * 100], color="black", s=80, zorder=6,
           label=f"Starting point: {EV_SHARE_NOW:.1%} ({EV_VEHICLES_NOW/1e6:.2f}M EVs)")
ax.axhline(EV_SHARE_NOW * 100, color="black", linewidth=0.8, linestyle=":", alpha=0.4)
for yr, label in [(2030, "2030 (PNEC)"), (2035, "2035 (EU ban)")]:
    ax.axvline(yr, color="gray", linewidth=1, linestyle="--", alpha=0.5)
    ax.text(yr + 0.15, 5, label, fontsize=8, color="gray")
ax.set_xlabel("Year"); ax.set_ylabel("EV fleet share (%)")
ax.set_title("EV adoption curves — three scenarios", fontsize=12)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.set_xlim(START_YEAR - 0.5, END_YEAR + 0.5)

ax2 = axes[1]
for name, (lbl, mul, color, ls) in scenarios.items():
    share = EV_SHARE_NOW * mul.values
    d_share = np.gradient(share, years_numeric)
    new_evs_M = d_share * TOTAL_FLEET / 1e6
    ax2.plot(years_numeric, new_evs_M, color=color, linestyle=ls, linewidth=2.5, label=lbl)
ax2.set_xlabel("Year"); ax2.set_ylabel("New EVs added per year (millions)")
ax2.set_title("Annual EV fleet increment by scenario", fontsize=12)
ax2.legend(fontsize=9, title="Scenario"); ax2.grid(True, alpha=0.3)
ax2.set_xlim(START_YEAR - 0.5, END_YEAR + 0.5); ax2.set_ylim(bottom=0)

fig.suptitle("French EV fleet adoption — market scenarios 2025-2040", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.savefig("outputs/03_adoption_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Panel national leger (tous departements, fenetre recente alignee sur des mois calendaires
# complets) : sert (a) de baseline d'energie ci-dessous et (b) de contexte historique dans le
# graphe de trajectoires plus bas. Charge une seule fois ici (supprime en fin de Part B) plutot
# que de garder le dataset complet (101 departements, ~9,5 ans) en memoire tout du long.
_raw_nat  = pd.read_pickle(DATA_PATH)
_dept_col = "insee_code_departement" if "insee_code_departement" in _raw_nat.columns else "department"
_ts_col   = "debut_session_timestamp" if "debut_session_timestamp" in _raw_nat.columns else "arrival_time"
_max_ts   = pd.to_datetime(_raw_nat[_ts_col], format="mixed", errors="coerce").max()
_last_full_month_end = _max_ts.to_period("M").to_timestamp() - pd.Timedelta(days=1)
_cutoff_nat = (_last_full_month_end - pd.DateOffset(months=NATIONAL_MONTHS)).replace(day=1)
_ts_all = pd.to_datetime(_raw_nat[_ts_col], format="mixed", errors="coerce")
_raw_nat = _raw_nat[(_ts_all >= _cutoff_nat) & (_ts_all <= _last_full_month_end)].copy()

df_nat = load_sessions(_raw_nat, verbose=False)
del _raw_nat, _ts_all
gc.collect()

panel_nat = build_panel(df_nat, freq="D", metric="energy_kwh")
panel_nat.index = pd.to_datetime(panel_nat.index)
print(f"Panel national (tous departements) : {panel_nat.shape[1]} departements, "
      f"{panel_nat.index.min().date()} -> {panel_nat.index.max().date()}  "
      f"({NATIONAL_MONTHS} mois calendaires complets)")
del df_nat
gc.collect()

last30 = panel_nat.tail(30)
baseline_energy_kwh = float(last30.sum(axis=1).mean())
print(f"Energy baseline (last 30-day mean, {panel_nat.shape[1]} departements): {baseline_energy_kwh:,.0f} kWh/day")

sim_start_date = panel.index.max() + pd.Timedelta(days=1)
print(f"Simulation start date: {sim_start_date.date()}")

hist_monthly_nat = panel_nat.sum(axis=1).resample("ME").mean()
monthly_returns  = hist_monthly_nat.pct_change().dropna()
historical_cv    = float(monthly_returns.std())
print(f"Historical monthly CV : {historical_cv:.3f}  (sigma du bruit log-normal par scenario)")

def build_lt_scenario_analytical(baseline_kwh, adoption_mul_values, sim_start, n_scenarios, cv, seed):
    '''
    Construit des trajectoires LT par scaling analytique de la courbe d'adoption -- elle-meme
    issue des fonctions gears.simulation.medium_term (Session 6), pas dupliquee a la main :
    energy(t, sc) = baseline * adoption_mul(t) * eps(t, sc), ou eps ~ LogNormal(0, cv) est un
    bruit multiplicatif mensuel, accumule comme marche aleatoire lissee (autocorrelee) pour des
    trajectoires visuellement coherentes.
    Analytique plutot que par simulation GMM session-par-session : a l'echelle nationale
    (jusqu'a plusieurs millions de sessions/jour en fin d'horizon sur les scenarios
    central/ambitieux), simuler chaque session individuellement sur 15 ans x 30 scenarios serait
    numeriquement intraitable dans le budget de ce notebook (voir REFACTOR_STATE.md, Session 6,
    pour la mesure de temps qui a motive ce choix -- MediumTermSimulator reste adapte a des
    horizons/echelles plus modestes, pas a ce cas d'usage national multi-decennal).
    '''
    rng = np.random.default_rng(seed)
    dates = pd.date_range(sim_start, periods=len(adoption_mul_values), freq="D")
    n_months = int(np.ceil(len(adoption_mul_values) / 30.44)) + 1
    rows = []
    for sc in range(n_scenarios):
        monthly_eps = np.exp(np.cumsum(rng.normal(0, cv / np.sqrt(12), n_months)))
        monthly_eps = monthly_eps / monthly_eps[0]
        month_idx = np.minimum((np.arange(len(dates)) / 30.44).astype(int), n_months - 1)
        energy = baseline_kwh * adoption_mul_values * monthly_eps[month_idx]
        rows.append(pd.DataFrame({
            "date": dates, "scenario": sc,
            "n_sessions": np.maximum(0, energy / 22).astype(int),
            "total_energy_kwh": np.maximum(0.0, energy),
        }))
    return pd.concat(rows, ignore_index=True)

N_LONG_SCENARIOS_ACTUAL = max(N_LONG_SCENARIOS, 30)
trajectory_results = {}
for name, (lbl, mul, color, ls) in scenarios.items():
    result = build_lt_scenario_analytical(
        baseline_kwh=baseline_energy_kwh, adoption_mul_values=mul.values, sim_start=sim_start_date,
        n_scenarios=N_LONG_SCENARIOS_ACTUAL, cv=historical_cv, seed=hash(lbl) % (2**31),
    )
    day1 = float(result[result["date"] == result["date"].min()]["total_energy_kwh"].mean())
    last_day = float(result[result["date"] == result["date"].max()]["total_energy_kwh"].mean())
    print(f"  {lbl}: day-1={day1:,.0f} kWh (baseline={baseline_energy_kwh:,.0f}) | "
          f"ratio_t0={day1/baseline_energy_kwh:.2f} | final={last_day/1e3:,.0f} MWh/day "
          f"(growth factor {last_day/day1:.1f}x)")
    trajectory_results[name] = {"result": result, "color": color, "ls": ls}

In [ ]:
# Session 6 : plot_lt_trajectories n'applique plus de clip artificiel a 10x l'ancre --
# voir gears/plotting.py et REFACTOR_STATE.md (AUDIT.md Mecanisme 2). Les scenarios central et
# ambitieux ci-dessous devraient maintenant continuer de croitre visiblement jusqu'a l'horizon,
# sans aplatissement avant 2040.
fig_lt = plot_lt_trajectories(
    panel=panel_nat,
    trajectory_results=trajectory_results,
    sim_start_date=sim_start_date,
    hist_tail_months=min(18, NATIONAL_MONTHS),
    zoom_days=30,
    energy_col="total_energy_kwh",
    figsize=(16, 7),
    title=f"Trajectoires long terme - scenarios adoption VE (moy. mensuelle, {LONG_YEARS} ans depuis {sim_start_date.year})",
    savepath="outputs/03_lt_trajectories.png",
)
plt.show()

del panel_nat
gc.collect()

In [ ]:
# Grid capacity implications, derivees directement du multiplicateur (fixe) de chaque scenario.
print("-- Grid capacity implications ------------------------------------------------")
base_sessions = EV_VEHICLES_NOW * SESSIONS_PER_EV_DAY
rows_cap = []
for name, (lbl, mul, color, ls) in scenarios.items():
    factor = float(mul.iloc[-1] / mul.iloc[0])
    sess_end = EV_SHARE_NOW * float(mul.iloc[-1]) * TOTAL_FLEET * SESSIONS_PER_EV_DAY
    e_start_gwh = baseline_energy_kwh * 1e-6
    e_end_gwh   = baseline_energy_kwh * factor * 1e-6
    print(f"  {name:24s}  {e_start_gwh:>8.2f} GWh -> {e_end_gwh:>8.2f} GWh  ({factor:>5.1f}x)")
    rows_cap.append({"Scenario": lbl, "Growth factor": round(factor, 1),
                     "Sessions end": int(sess_end),
                     f"Energy {END_YEAR} (GWh/day)": round(e_end_gwh, 2)})
cap_df = pd.DataFrame(rows_cap)
print()
print(cap_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
scen_labels  = [lbl for _, (lbl, _, _, _) in scenarios.items()]
colors_list  = [color for _, (_, _, color, _) in scenarios.items()]

ax = axes[0]
sessions_end = [row["Sessions end"] for row in rows_cap]
bars = ax.bar(scen_labels, [s/1e6 for s in sessions_end], color=colors_list,
              alpha=0.85, edgecolor="white", linewidth=1.5, width=0.5)
ax.axhline(base_sessions/1e6, color="black", linewidth=1.5, linestyle="--",
           label=f"Baseline {START_YEAR} ({base_sessions/1e6:.2f}M sess/day)")
for bar, v in zip(bars, sessions_end):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
            f"{v/1e6:.2f}M", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_ylabel("Daily sessions (millions)")
ax.set_title(f"Projected daily sessions in {END_YEAR}", fontsize=11)
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)

ax2 = axes[1]
e_end = [row[f"Energy {END_YEAR} (GWh/day)"] for row in rows_cap]
bars2 = ax2.bar(scen_labels, e_end, color=colors_list,
                alpha=0.85, edgecolor="white", linewidth=1.5, width=0.5)
ax2.axhline(baseline_energy_kwh/1e6, color="black", linewidth=1.5, linestyle="--",
            label=f"Baseline {START_YEAR} ({baseline_energy_kwh/1e6:.2f} GWh/day)")
for bar, v in zip(bars2, e_end):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
             f"{v:.1f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax2.set_ylabel("Daily energy (GWh/day)")
ax2.set_title(f"Projected daily energy in {END_YEAR}", fontsize=11)
ax2.legend(fontsize=9); ax2.grid(axis="y", alpha=0.3)

fig.suptitle(f"Grid capacity implications -- {START_YEAR} vs. {END_YEAR}", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.savefig("outputs/03_capacity_2040.png", dpi=150, bbox_inches="tight")
plt.show()

---
# Section C — Reconstruction des profils de charge avec smart charging

Cette section utilise `OutputAggregator.build_load_profiles()` pour reconstruire un profil de charge horaire annuel à partir du GMM français, puis compare le comportement **plug-and-charge** (charge immédiate à l'arrivée) avec le **smart charging V1G** (report de la charge vers les heures de moindre coût), grâce à `SmartChargingOptimizer`.

Le signal de prix utilisé ci-dessous est **synthétique** (sinusoïde journalière + bruit) : il permet d'illustrer la méthode sans disposer d'un fichier de prix réel. Remplacer `price_signal` par une série issue de l'EPEX Spot ou du gestionnaire de réseau pour une analyse opérationnelle.

In [ ]:
# Reconstruction "plug-and-charge" (sans smart charging) -- bundle COMPLET, illustration
# nationale. Rapide (~20-25s) meme sur les 8008 contextes du GMM car aucun appel a
# l'optimiseur n'est necessaire ici (voir cellule suivante pour la partie couteuse).
registry = NativeGMMRegistry()
gmm = registry.load("french")
vae = registry.load("french_vae_sample")
print(f"GMM  'french'            : {len(gmm.models_)} contextes")
print(f"VAE  'french_vae_sample' : {len(vae.models_)} contextes (bundle sample, is_sample_={vae.is_sample_})")

agg = OutputAggregator(resolution_min=60)
loc_map = LOCATION_POWER_PRESETS[PROFILE_PRESET]

result_plug_gmm = agg.build_load_profiles(
    gmm=gmm, year=PROFILE_YEAR, n_days_mc=N_DAYS_MC_PROFILES,
    charging_mode="by_location", location_power_map=loc_map, seed=42,
)
result_plug_vae = agg.build_load_profiles(
    gmm=vae, year=PROFILE_YEAR, n_days_mc=N_DAYS_MC_PROFILES,
    charging_mode="by_location", location_power_map=loc_map, seed=42,
)
ts_plug_gmm = result_plug_gmm["ts"]
ts_plug_vae = result_plug_vae["ts"]

print(f"\n-- Profil plug-and-charge annuel (bundle complet) -----------------------")
print(f"GMM : pic {ts_plug_gmm.max():,.0f} kW | moyenne {ts_plug_gmm.mean():,.0f} kW")
print(f"VAE : pic {ts_plug_vae.max():,.0f} kW | moyenne {ts_plug_vae.mean():,.0f} kW")

fig, ax = plt.subplots(figsize=(14, 4))
week = slice(0, 24*7)
ax.plot(ts_plug_gmm.index[week], ts_plug_gmm.values[week], label="GMM (french)", color="#2E86AB", linewidth=1.5)
ax.plot(ts_plug_vae.index[week], ts_plug_vae.values[week], label="VAE (french_vae_sample)", color="#F4A261", linewidth=1.5, linestyle="--")
ax.set_ylabel("Puissance appelee (kW)"); ax.set_title("Profil de charge plug-and-charge -- premiere semaine, GMM vs VAE (bundle complet)")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig("outputs/03_plug_profile_gmm_vs_vae.png", dpi=150, bbox_inches="tight")
plt.show()

### C.2 - Smart charging (V1G) - reconstruction restreinte (GMM + VAE)

**Cette section remplace une cellule qui, sur le bundle complet (8008 contextes), prenait
plus de 30 minutes** : `OutputAggregator._build_smart_charging_ts` appelle
`SmartChargingOptimizer.optimise()` une fois PAR CONTEXTE (pas par type de journee), et le
bundle `"french"` complet en a 8008.

Reduction appliquee (illustrative, pas une analyse de production) :
- contextes restreints a `SMART_LOC_TYPES` x `SMART_DEPTS` (~110 contextes au lieu de 8008)
- `N_DAYS_MC_SMART` reduit a 5 jours Monte-Carlo
- signal de prix limite a 2 semaines (le motif journalier obtenu est ensuite reconduit sur
  l'annee complete par `OutputAggregator` -- la saisonnalite du prix/comportement n'est donc
  pas modelisee ici, uniquement le contraste heures pleines/creuses)

`build_load_profiles(..., smart_charging_signal=...)` retourne a la fois `"ts"` (plug) et
`"ts_smart"` (V1G) calcules sur le MEME sous-ensemble de contextes -- la comparaison
plug-vs-smart ci-dessous est donc bien a perimetre constant (contrairement au profil "plug"
national de C.1, qui sert uniquement d'illustration a plus grande echelle).

In [ ]:
def reduce_gmm_contexts(model, keep_loc, keep_dept):
    '''Sous-ensemble d'un EVSessionGMM/VAE deja fitte, en filtrant les contextes.'''
    keep_ctx = {
        ctx: m for ctx, m in model.models_.items()
        if dict(zip(model.stratify_by, ctx))["location_type"] in keep_loc
        and dict(zip(model.stratify_by, ctx))["department"] in keep_dept
    }
    small = EVSessionGMM(stratify_by=model.stratify_by, random_state=42)
    small.models_ = keep_ctx
    small.n_sessions_per_day_ = {k: v for k, v in model.n_sessions_per_day_.items() if k in keep_ctx}
    small.context_counts_ = {k: v for k, v in model.context_counts_.items() if k in keep_ctx}
    small.is_fitted_ = True
    return small

gmm_smart_subset = reduce_gmm_contexts(gmm, SMART_LOC_TYPES, SMART_DEPTS)
vae_smart_subset = reduce_gmm_contexts(vae, SMART_LOC_TYPES, SMART_DEPTS)
print(f"Contextes retenus -- GMM: {len(gmm_smart_subset.models_)} | VAE: {len(vae_smart_subset.models_)} "
      f"(lieux={sorted(SMART_LOC_TYPES)}, depts={sorted(SMART_DEPTS)})")

# Signal de prix jour/nuit tres simplifie, 2 semaines seulement (voir note ci-dessus)
_idx_price = pd.date_range(f"{PROFILE_YEAR}-01-01", periods=24 * 14, freq="h")
_rng_price = np.random.default_rng(0)
_hours = np.arange(len(_idx_price))
_daily_pattern = 0.5 - 0.4 * np.cos(2 * np.pi * (_hours % 24 - 3) / 24)
_noise = _rng_price.normal(0, 0.05, size=len(_idx_price))
price_signal = pd.Series(np.clip(_daily_pattern + _noise, 0.02, 1.0), index=_idx_price, name="price_eur_kwh")

result_smart_gmm = agg.build_load_profiles(
    gmm=gmm_smart_subset, year=PROFILE_YEAR, n_days_mc=N_DAYS_MC_SMART,
    charging_mode="mean_power", location_power_map=loc_map, seed=42,
    smart_charging_signal=price_signal,
)
result_smart_vae = agg.build_load_profiles(
    gmm=vae_smart_subset, year=PROFILE_YEAR, n_days_mc=N_DAYS_MC_SMART,
    charging_mode="mean_power", location_power_map=loc_map, seed=42,
    smart_charging_signal=price_signal,
)
print("\nReconstruction terminee (voir figure ci-dessous).")

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
week = slice(0, 24 * 7)

for ax, result, label in zip(
    axes, [result_smart_gmm, result_smart_vae], ["GMM", "VAE"]
):
    ts_plug_sub  = result["ts"]
    ts_smart_sub = result["ts_smart"]
    ax.plot(ts_plug_sub.index[week], ts_plug_sub.values[week], label="Plug-and-charge", color="#9B5DE5", linewidth=1.8)
    ax.plot(ts_smart_sub.index[week], ts_smart_sub.values[week], label="Smart charging (V1G)", color="#3BB273", linewidth=1.8, linestyle="--")
    ax.set_ylabel("Puissance (kW)")
    ax.set_title(f"{label} -- work+home, depts {sorted(SMART_DEPTS)} -- premiere semaine", fontsize=11)
    ax.legend(); ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("")
fig.suptitle("Plug-and-charge vs. smart charging (V1G) -- GMM vs VAE, meme perimetre restreint",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.savefig("outputs/03_plug_vs_smart_gmm_vs_vae.png", dpi=150, bbox_inches="tight")
plt.show()

for result, label in zip([result_smart_gmm, result_smart_vae], ["GMM", "VAE"]):
    mean_shift = 1 - result["ts_smart"].mean() / result["ts"].mean()
    energy_shift = 1 - result["ts_smart"].sum() / result["ts"].sum()
    print(f"{label}: variation puissance moyenne avec V1G = {mean_shift:+.2%} | "
          f"variation energie totale = {energy_shift:+.2%}")
print("\n(Sur ce sous-ensemble restreint et ce signal de prix simplifie, l'effet sur la SEULE\n"
      " puissance de pointe horaire peut rester nul si l'heure de pointe est dominee par des\n"
      " sessions peu flexibles -- voir savings_summary() ci-dessous pour l'effet cout, plus robuste.)")

In [ ]:
opt_profiles = SmartChargingOptimizer(signal_type="price", resolution_min=60)

for result, model, label in zip(
    [result_smart_gmm, result_smart_vae], [gmm_smart_subset, vae_smart_subset], ["GMM", "VAE"]
):
    if "savings_summary" in result:
        summary = result["savings_summary"]
    else:
        # Reconstruit un echantillon de sessions representatif pour calculer les economies --
        # NB: le parametre s'appelle `context=`, pas `ctx=`.
        first_ctx_key = next(iter(model.models_.keys()))
        first_ctx = dict(zip(model.stratify_by, first_ctx_key))
        sample_sessions = model.sample(
            n_sessions=200, context=first_ctx, seed=1,
        )
        sample_sessions["arrival_time"] = pd.Timestamp(f"{PROFILE_YEAR}-01-06") + pd.to_timedelta(
            sample_sessions["arrival_hour"], unit="h"
        )
        sample_sessions["power_kw"] = 7.4
        sample_opt = opt_profiles.optimise(sample_sessions, price_signal)
        summary = opt_profiles.savings_summary(sample_opt)

    print(f"-- {label} -- savings_summary() ({first_ctx if 'savings_summary' not in result else 'agrege'}) --")
    for k, v in summary.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
    print()

## Bilan

### Section A — Moyen terme
- `DepartmentForecaster` (SARIMA) évalué sur une fenêtre de test ; voir le tableau MAPE ci-dessus.
- Fan charts avec IC à 80 %, dont la largeur mesurée (imprimée ci-dessus) reflète désormais
  l'écart-type historique complet plutôt qu'une fraction de celui-ci (Session 6 — voir
  `REFACTOR_STATE.md`, AUDIT.md Mécanisme 1).

### Section B — Long terme

Les trois scénarios (conservateur / central / ambitieux) sont construits à partir des fonctions
`linear_growth_profile`, `s_curve_growth_profile` et `bass_diffusion_profile` de
`gears.simulation.medium_term`, chacune calibrée pour approcher une part de flotte cible en
2040 tirée d'Avere-France / SNEF / objectifs UE-PNEC (voir la cellule de définition des
scénarios). Le facteur de croissance 2025→2040 réellement obtenu pour chaque
scénario est imprimé ci-dessus (mesuré directement depuis les fonctions, pas fixé a priori) ;
les trajectoires de la figure "Trajectoires long terme" ne sont plus aplaties artificiellement
avant l'horizon (Session 6 — AUDIT.md Mécanisme 2).

**Points clés :** point de départ = parc observé actuel (3,1 %, ~1,19 M VE) ; ≥ 30 scénarios
Monte-Carlo rééchantillonnés mensuellement pour des fan charts lisses.

### Section C — Profils de charge et smart charging
- `OutputAggregator.build_load_profiles()` reconstruit le profil horaire annuel à partir du GMM français, avec mix de puissance par type de lieu (preset `LOCATION_POWER_PRESETS`).
- Le smart charging V1G déplace la charge vers les heures creuses du signal de prix ; l'effet mesuré sur la puissance moyenne et l'énergie totale est imprimé ci-dessus pour le GMM et le VAE.
- **Pour une application réelle** : remplacer le signal synthétique par des données EPEX Spot ou RTE eCO2mix (résolution horaire ou 30 min).

## Bilan (complement GMM vs VAE)

| Aspect | GMM (`french`) | VAE (`french_vae_sample`) |
|---|---|---|
| Contextes (bundle complet) | 8008 | 516 (top-5 departements, `is_sample_=True`) |
| Reconstruction "plug" (bundle complet) | ~20-25s pour 8 jours MC | plus rapide encore (moins de contextes) |
| Reconstruction "smart charging" (bundle complet) | **>30 minutes** (8008 x `optimise()`) | non testee a cette echelle -- bundle sample deja restreint |
| Reconstruction "smart charging" (sous-ensemble ~110 ctx) | mesure imprimee ci-dessus | mesure imprimee ci-dessus |
| `.sample()`, `plot_marginals`, `bic_summary`, `get_sklearn_gmm` | identiques (voir Notebook 1) | identiques (voir Notebook 1) |
| `savings_summary()` | fonctionne apres correctif `context=` | idem |

**Point cle sur la Section C :** le goulot d'etranglement n'est PAS le nombre de jours
Monte-Carlo mais le nombre de CONTEXTES du bundle (chaque contexte declenche un appel
`SmartChargingOptimizer.optimise()` independant). Reduire les types de lieu et les
departements a l'echelle du contexte est donc l'unique levier vraiment efficace -- reduire
uniquement `n_days_mc` ou raccourcir le signal de prix n'aurait pas suffi a lui seul.